# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamKottish/FlyRankML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

# Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token: "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MAR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-03/*.parquet')"
)

FACT_APR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-04/*.parquet')"
)

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Connected")
print("Feature window: March 2026")
print("Outcome/evaluation window: April 2026")

Paste your Hugging Face READ token: ··········
✓ Connected
Feature window: March 2026
Outcome/evaluation window: April 2026


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


My baseline asks a simple question: **which visible pages had lower CTR than comparable pages at similar search positions during March 2026?**

First, I require at least 500 March impressions so very low-volume pages do not dominate the queue. I compare each page's CTR with the median CTR of pages in the same position band. The baseline score increases when the page has a larger positive CTR gap and more search exposure.

The transparent rule is:

**baseline score = positive position-adjusted CTR gap × log(1 + March impressions)**

The raw score is normalized to 0–100 only for readability. No model is trained and no weights are fitted from the April outcome.

Reason codes include:

- `below_position_peer_ctr` — CTR is at least 0.10 percentage points below the median of its position band.
- `high_visibility` — at least 3,000 impressions in March.
- `strong_position_ctr_gap` — average position is 20 or better but CTR is below comparable pages.
- `volatile_position` — search position varied substantially during March.
- `monitor_only` — the page does not show a strong enough CTR gap for immediate review.

The intended action for strong candidates is to review the search snippet, title/meta, and intent match. This is decision-support only; the rule does not prove that an edit will improve CTR.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------
# March baseline data
# ------------------------------------------------------

march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS feature_impressions,
        SUM(gsc_clicks) AS feature_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        END AS feature_ctr,

        AVG(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS feature_avg_position,

        STDDEV_SAMP(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS feature_position_std

    FROM {FACT_MAR}

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) >= 500
""").df()

march["feature_position_std"] = (
    march["feature_position_std"].fillna(0)
)

# Remove rows where position cannot be measured
baseline_df = march[
    march["feature_avg_position"].notna()
].copy()


# Position bands
def position_band(pos):
    if pos <= 3:
        return "top_3"
    elif pos <= 10:
        return "page_1"
    elif pos <= 20:
        return "striking"
    elif pos <= 50:
        return "page_3_5"
    return "deep"


baseline_df["position_band"] = (
    baseline_df["feature_avg_position"]
    .apply(position_band)
)

# Expected CTR = median among pages in same position band
baseline_df["band_median_ctr"] = (
    baseline_df
    .groupby("position_band")["feature_ctr"]
    .transform("median")
)

baseline_df["ctr_gap_pp"] = (
    baseline_df["band_median_ctr"]
    - baseline_df["feature_ctr"]
)

# Only positive underperformance contributes to score
baseline_df["positive_ctr_gap"] = (
    baseline_df["ctr_gap_pp"].clip(lower=0)
)

# Transparent, unfitted baseline
baseline_df["raw_baseline_score"] = (
    baseline_df["positive_ctr_gap"]
    * np.log1p(baseline_df["feature_impressions"])
)

max_score = baseline_df["raw_baseline_score"].max()

if max_score > 0:
    baseline_df["baseline_action_score"] = (
        100
        * baseline_df["raw_baseline_score"]
        / max_score
    )
else:
    baseline_df["baseline_action_score"] = 0.0


# Reason codes
def make_reasons(row):
    reasons = []

    if row["ctr_gap_pp"] >= 0.10:
        reasons.append("below_position_peer_ctr")

    if row["feature_impressions"] >= 3000:
        reasons.append("high_visibility")

    if (
        row["feature_avg_position"] <= 20
        and row["ctr_gap_pp"] >= 0.10
    ):
        reasons.append("strong_position_ctr_gap")

    if row["feature_position_std"] >= 10:
        reasons.append("volatile_position")

    if not reasons:
        reasons.append("monitor_only")

    return "; ".join(reasons)


baseline_df["reason_codes"] = baseline_df.apply(
    make_reasons,
    axis=1
)

print(f"Baseline population: {len(baseline_df):,} pages")

display(
    baseline_df[
        [
            "feature_impressions",
            "feature_ctr",
            "feature_avg_position",
            "position_band",
            "band_median_ctr",
            "ctr_gap_pp",
            "baseline_action_score",
            "reason_codes",
        ]
    ].head()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline population: 61,924 pages


,feature_impressions,feature_ctr,feature_avg_position,position_band,band_median_ctr,ctr_gap_pp,baseline_action_score,reason_codes
0,6523.0,0.107313,7.209549,page_1,0.216857,0.109544,35.254015,below_position_peer_ctr; high_visibility; stro...
1,5630.0,0.106572,6.724039,page_1,0.216857,0.110285,34.897548,below_position_peer_ctr; high_visibility; stro...
2,4944.0,0.262945,7.244844,page_1,0.216857,-0.046088,0.000000,high_visibility
3,7709.0,0.259437,5.258331,page_1,0.216857,-0.042580,0.000000,high_visibility
4,3561.0,0.280820,8.834415,page_1,0.216857,-0.063963,0.000000,high_visibility


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


I rank every eligible March page from highest to lowest baseline action score.

The queue includes the pseudonymized content identifier, the measured March signals, the action score, reason codes, a suggested action, and a confidence note.

The score itself uses only March information. April measurements are not used to determine the rank.

The ranked queue is written to:

`work/outputs/baseline_action_score.csv`

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Suggested action
def choose_action(row):
    if row["ctr_gap_pp"] >= 0.10:
        if row["feature_avg_position"] <= 20:
            return "review_title_snippet_and_intent"
        return "review_search_snippet"

    return "monitor"


# Confidence
def confidence_note(row):
    if (
        row["feature_impressions"] >= 3000
        and row["ctr_gap_pp"] >= 0.20
    ):
        return "high"

    if (
        row["feature_impressions"] >= 1000
        or row["ctr_gap_pp"] >= 0.10
    ):
        return "medium"

    return "low"


# What could make this recommendation wrong?
def wrong_reason(row):
    if row["feature_position_std"] >= 10:
        return (
            "Position was volatile; the CTR gap may reflect "
            "ranking movement rather than a persistent opportunity."
        )

    if row["feature_impressions"] < 1000:
        return (
            "Evidence volume is limited; a small number of clicks "
            "could move CTR materially."
        )

    return (
        "Seasonality, SERP changes, intent differences, or later "
        "position changes could explain the observed CTR gap."
    )


baseline_df["action"] = baseline_df.apply(
    choose_action,
    axis=1
)

baseline_df["confidence"] = baseline_df.apply(
    confidence_note,
    axis=1
)

baseline_df["what_could_make_it_wrong"] = baseline_df.apply(
    wrong_reason,
    axis=1
)


baseline_queue = (
    baseline_df
    .sort_values(
        ["baseline_action_score", "feature_impressions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

baseline_queue.insert(
    0,
    "rank",
    np.arange(1, len(baseline_queue) + 1)
)


queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_action_score",
    "action",
    "reason_codes",
    "confidence",
    "what_could_make_it_wrong",
    "feature_impressions",
    "feature_clicks",
    "feature_ctr",
    "feature_avg_position",
    "feature_position_std",
    "position_band",
    "band_median_ctr",
    "ctr_gap_pp",
]

baseline_queue = baseline_queue[queue_columns]

OUTPUT_PATH = OUTPUT_DIR / "baseline_action_score.csv"

baseline_queue.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"✓ Ranked {len(baseline_queue):,} pages")
print(f"✓ CSV written to: {OUTPUT_PATH}")

display(baseline_queue.head(10))

✓ Ranked 61,924 pages
✓ CSV written to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,baseline_action_score,action,reason_codes,confidence,what_could_make_it_wrong,feature_impressions,feature_clicks,feature_ctr,feature_avg_position,feature_position_std,position_band,band_median_ctr,ctr_gap_pp
0,1,client_1a730cb2640a1abf,content_d61fc394d10cba41,100.000000,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",38000.0,1.0,0.002632,2.740744,2.044146,top_3,0.261438,0.258806
1,2,client_fef1a8f436438636,content_66bf45eb0c5bb550,95.192817,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",24259.0,1.0,0.004122,2.784944,1.506762,top_3,0.261438,0.257316
2,3,client_73cda7b4e4f265ea,content_8e1334d6356668e3,93.542270,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",134984.0,1.0,0.000741,4.545582,2.576839,page_1,0.216857,0.216116
3,4,client_62f4a7e64f5e0096,content_fc67675904376267,93.357235,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",60172.0,18.0,0.029914,2.261303,0.523544,top_3,0.261438,0.231524
4,5,client_73cda7b4e4f265ea,content_b9acd1ebff7d34ff,93.053223,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",25941.0,3.0,0.011565,2.418270,0.900653,top_3,0.261438,0.249873
5,6,client_73cda7b4e4f265ea,content_fec55986a1868d62,92.846979,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",124075.0,1.0,0.000806,9.385150,9.811790,page_1,0.216857,0.216051
6,7,client_23a62021009f63c4,content_44f34c0a90047651,92.386634,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",212404.0,24.0,0.011299,7.346909,4.565167,page_1,0.216857,0.205558
7,8,client_a80fca3f171ed1de,content_fa17add7836d36c3,90.433753,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",12588.0,0.0,0.000000,1.902457,1.579138,top_3,0.261438,0.261438
8,9,client_e547b89c05043229,content_306bc78dff1eb683,90.315447,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",80821.0,35.0,0.043306,1.488604,0.626193,top_3,0.261438,0.218132
9,10,client_73cda7b4e4f265ea,content_1d7764b642f7bb9f,90.072493,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",23402.0,4.0,0.017093,1.843455,1.715716,top_3,0.261438,0.244345


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I manually inspect the highest-ranked 20 recommendations rather than trusting the numerical score automatically.

For each page, I review:

- the recommended action,
- the reason codes,
- the confidence level,
- the amount of March evidence,
- and what could make the recommendation wrong.

For retrospective evaluation only, I also compare the baseline ranking with the April CTR-opportunity proxy. April is never used to create the March score.

The primary baseline metric is **Precision@20**: among the first 20 evaluable recommendations, what proportion meet the later April opportunity definition? I also report the April proxy base rate so the baseline can be compared with the rate expected without ranking.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------
# April observed outcome
# ------------------------------------------------------

april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS outcome_impressions,
        SUM(gsc_clicks) AS outcome_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        END AS outcome_ctr,

        AVG(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS outcome_avg_position

    FROM {FACT_APR}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()


# Only rows with enough April evidence can receive the
# retrospective CTR opportunity label.
april_eval = april[
    (april["outcome_impressions"] >= 500)
    & april["outcome_avg_position"].notna()
].copy()

april_eval["outcome_position_band"] = (
    april_eval["outcome_avg_position"]
    .apply(position_band)
)

april_eval["outcome_band_median_ctr"] = (
    april_eval
    .groupby("outcome_position_band")["outcome_ctr"]
    .transform("median")
)

april_eval["outcome_ctr_gap_pp"] = (
    april_eval["outcome_band_median_ctr"]
    - april_eval["outcome_ctr"]
)

april_eval["opportunity_proxy"] = (
    april_eval["outcome_ctr_gap_pp"] > 0.10
).astype(int)


# ------------------------------------------------------
# Retrospective evaluation population
# ------------------------------------------------------

evaluation_df = baseline_queue.merge(
    april_eval[
        [
            "client_hash_id",
            "content_hash_id",
            "outcome_impressions",
            "outcome_ctr",
            "outcome_position_band",
            "outcome_ctr_gap_pp",
            "opportunity_proxy",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

evaluation_df = evaluation_df.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)


K = 20

eval_top20 = evaluation_df.head(K).copy()

precision_at_20 = (
    eval_top20["opportunity_proxy"].mean()
)

base_rate = (
    evaluation_df["opportunity_proxy"].mean()
)


print(f"Evaluable pages: {len(evaluation_df):,}")
print(f"April opportunity base rate: {base_rate:.3f}")
print(f"Baseline Precision@20: {precision_at_20:.3f}")

print(
    f"\nTop 20 correct by the April proxy: "
    f"{eval_top20['opportunity_proxy'].sum()} of {len(eval_top20)}"
)


# ------------------------------------------------------
# Human-readable top-20
# ------------------------------------------------------

top20_review = eval_top20[
    [
        "rank",
        "content_hash_id",
        "baseline_action_score",
        "action",
        "reason_codes",
        "confidence",
        "what_could_make_it_wrong",
        "feature_impressions",
        "feature_ctr",
        "feature_avg_position",
        "ctr_gap_pp",
        "opportunity_proxy",
    ]
].copy()

top20_review["april_result"] = np.where(
    top20_review["opportunity_proxy"] == 1,
    "supported_by_April_proxy",
    "not_supported_by_April_proxy"
)

display(top20_review)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Evaluable pages: 51,496
April opportunity base rate: 0.240
Baseline Precision@20: 0.950

Top 20 correct by the April proxy: 19 of 20


,rank,content_hash_id,baseline_action_score,action,reason_codes,confidence,what_could_make_it_wrong,feature_impressions,feature_ctr,feature_avg_position,ctr_gap_pp,opportunity_proxy,april_result
0,1,content_d61fc394d10cba41,100.000000,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",38000.0,0.002632,2.740744,0.258806,1,supported_by_April_proxy
1,2,content_66bf45eb0c5bb550,95.192817,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",24259.0,0.004122,2.784944,0.257316,1,supported_by_April_proxy
2,3,content_8e1334d6356668e3,93.542270,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",134984.0,0.000741,4.545582,0.216116,1,supported_by_April_proxy
3,4,content_fc67675904376267,93.357235,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",60172.0,0.029914,2.261303,0.231524,1,supported_by_April_proxy
4,5,content_b9acd1ebff7d34ff,93.053223,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",25941.0,0.011565,2.418270,0.249873,1,supported_by_April_proxy
5,6,content_fec55986a1868d62,92.846979,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",124075.0,0.000806,9.385150,0.216051,1,supported_by_April_proxy
6,7,content_44f34c0a90047651,92.386634,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",212404.0,0.011299,7.346909,0.205558,1,supported_by_April_proxy
7,9,content_306bc78dff1eb683,90.315447,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",80821.0,0.043306,1.488604,0.218132,1,supported_by_April_proxy
8,10,content_1d7764b642f7bb9f,90.072493,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",23402.0,0.017093,1.843455,0.244345,1,supported_by_April_proxy
9,11,content_cd3d932d4e1c8db0,88.712599,review_title_snippet_and_intent,below_position_peer_ctr; high_visibility; stro...,high,"Seasonality, SERP changes, intent differences,...",89332.0,0.004478,7.786219,0.212379,1,supported_by_April_proxy


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


I do not assume that every highly ranked recommendation is correct.

A weak pick can occur when:

- the page does not meet the later April opportunity proxy,
- March position was unusually volatile,
- the CTR gap was small,
- the amount of evidence was limited,
- or search behavior changed between March and April.

A false positive does not necessarily mean the baseline logic is useless. It may indicate seasonality, SERP changes, changes in ranking position, different search intent, or ordinary measurement noise.

I also verify that the ranking score contains no April outcome fields, no opportunity-proxy fields, no product decision flags, and no private raw identifiers. `client_hash_id` and `content_hash_id` are retained only as context fields and are not part of the numerical score.

One limitation of this retrospective evaluation is that the April evaluation population requires at least 500 April impressions so that the opportunity proxy is measurable. This uses future information only to define which rows can be evaluated; it is not used to calculate the March baseline score or deployment queue.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------
# Weak picks from the evaluated Top 20
# ------------------------------------------------------

weak_picks = eval_top20[
    (eval_top20["opportunity_proxy"] == 0)
    | (eval_top20["confidence"] == "low")
    | (eval_top20["feature_position_std"] >= 10)
].copy()

print(f"Weak/questionable picks found: {len(weak_picks)}")

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "baseline_action_score",
            "reason_codes",
            "confidence",
            "feature_position_std",
            "ctr_gap_pp",
            "outcome_ctr_gap_pp",
            "opportunity_proxy",
            "what_could_make_it_wrong",
        ]
    ]
)


# ------------------------------------------------------
# Leakage check
# ------------------------------------------------------

SCORING_COLUMNS = {
    "feature_impressions",
    "feature_ctr",
    "feature_avg_position",
    "feature_position_std",
    "position_band",
    "band_median_ctr",
    "ctr_gap_pp",
    "positive_ctr_gap",
    "raw_baseline_score",
}

future_terms = [
    "outcome",
    "april",
    "proxy",
]

future_leaks = [
    col for col in SCORING_COLUMNS
    if any(term in col.lower() for term in future_terms)
]


product_flags = {
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "is_quick_win",
    "refresh_flag",
}

product_leaks = sorted(
    SCORING_COLUMNS.intersection(product_flags)
)


private_fields = {
    "client_name",
    "domain",
    "raw_url",
    "url",
    "raw_query",
    "query",
    "content_title",
}

privacy_leaks = sorted(
    SCORING_COLUMNS.intersection(private_fields)
)


print("\nFuture/label leaks:", future_leaks)
print("Product decision leaks:", product_leaks)
print("Private-field leaks:", privacy_leaks)

assert future_leaks == []
assert product_leaks == []
assert privacy_leaks == []

print("\n✓ Baseline score uses March observable signals only.")
print("✓ April data is used only for retrospective evaluation.")
print("✓ No product decision flags enter the score.")
print("✓ No raw client names, URLs, titles, or queries are used.")

Weak/questionable picks found: 1


,rank,content_hash_id,baseline_action_score,reason_codes,confidence,feature_position_std,ctr_gap_pp,outcome_ctr_gap_pp,opportunity_proxy,what_could_make_it_wrong
18,20,content_bf078007df823490,85.082743,below_position_peer_ctr; high_visibility; stro...,high,4.650569,0.216857,-0.011466,0,"Seasonality, SERP changes, intent differences,..."



Future/label leaks: []
Product decision leaks: []
Private-field leaks: []

✓ Baseline score uses March observable signals only.
✓ April data is used only for retrospective evaluation.
✓ No product decision flags enter the score.
✓ No raw client names, URLs, titles, or queries are used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.